In [ ]:
import numpy as np
import pandas as pd

from bert_hacking.storage import load_regressions

# One row per (run, policy, hypothesis), with the run's config and its test macro-F1 for
# that policy. Chunk runs have two policies ("any", "mean"), truncate runs one ("trunc"):
# risks are computed for one policy at a time, never across policies.
df_all = load_regressions()
POLICY = "mean"
df = df_all[df_all["policy"] == POLICY].copy()
df["score_on_test-bins"] = df.score_on_test.apply(lambda x: float(f"{x:.1f}"))
df


In [ ]:
# Hypotheses where a regression could not be compared (e.g. constant predicted labels)
df["error"].value_counts() if "error" in df else "no failed comparison"


In [ ]:
df.columns

In [ ]:
import numpy as np 
# Risk type 1
df_not_na = df.loc[np.logical_and(
    df.h_t.notna(),
    df.error_type_1.notna(), 
)]
df_not_na_h_0 = df_not_na[df_not_na.h_t == False]

risk_type_I_per_task = []
for task, task_df in df_not_na_h_0.groupby("task"):
    risk_type_I_per_hyp = []
    for hyp, task_hyp_df in task_df.groupby("hypothesis"):
        risk_type_I_per_config = []
        for config, task_hyp_config in task_hyp_df.groupby("run_id"):
            risk_type_I_per_config += [task_hyp_config.error_type_1.mean()]
        risk_type_I_per_hyp += [np.mean(risk_type_I_per_config)]
    risk_type_I_per_task += [np.mean(risk_type_I_per_hyp)]
print(f"Risk type I: {np.mean(risk_type_I_per_task)}")    

In [ ]:
def get_risk(df: pd.DataFrame, type:str) -> float:
    df_out = df.copy()
    df_out = df.loc[np.logical_and(
        df.h_t.notna(),
        df[type].notna(), 
    )]
    if type == 'error_type_1':
        df_out = df_out[df_out.h_t == False]
    elif type in ['error_type_2', 'error_type_S']:
        df_out = df_out[df_out.h_t == True]
    else: 
        return -1

    return (
        df_out
        .groupby(["task", "hypothesis", "run_id"], as_index=False)[type].mean()
        .groupby(["task", "hypothesis"], as_index=False)[type].mean()
        .groupby(["task"], as_index=False)[type].mean()
        [type].mean()
    )

def get_risk_support(df: pd.DataFrame, type:str) -> float:
    df_out = df.copy()
    df_out = df.loc[np.logical_and(
        df.h_t.notna(),
        df[type].notna(), 
    )]
    if type == 'error_type_1':
        df_out = df_out[df_out.h_t == False]
    elif type in ['error_type_2', 'error_type_S']:
        df_out = df_out[df_out.h_t == True]
    else: 
        return -1
    return len(df_out)

print(f"Risk type I : {get_risk(df, 'error_type_1'):.4f} (support: {get_risk_support(df, 'error_type_1'):>8})")
print(f"Risk type II: {get_risk(df, 'error_type_2'):.4f} (support: {get_risk_support(df, 'error_type_2'):>8})")
print(f"Risk type S : {get_risk(df, 'error_type_S'):.4f} (support: {get_risk_support(df, 'error_type_S'):>8})")

In [ ]:
# Effect of the long-text policy (any / mean / trunc) on the risks
pd.DataFrame([
    {
        "policy": policy,
        **{f"Risk type {t}": get_risk(sub_df, f"error_type_{t}") for t in ("1", "2", "S")},
        **{f"Risk type {t} (support)": get_risk_support(sub_df, f"error_type_{t}") for t in ("1", "2", "S")},
    }
    for policy, sub_df in df_all.groupby("policy")
])


In [ ]:
import plotly.express as px


result = []
for score, sub_df in df.groupby("score_on_test-bins"):
    result += [{
        "score": score,
        "Risk type 1": get_risk(sub_df, "error_type_1"),
        "Risk type 1 (support)": get_risk_support(sub_df, "error_type_1"),
        "Risk type 2": get_risk(sub_df, "error_type_2"),
        "Risk type 2 (support)": get_risk_support(sub_df, "error_type_2"),
        "Risk type S": get_risk(sub_df, "error_type_S"),
        "Risk type S (support)": get_risk_support(sub_df, "error_type_S"),
    }]
result = pd.DataFrame(result)

plot_df = result.melt(id_vars=["score"], value_vars=["Risk type 1", "Risk type 2", "Risk type S"])
px.bar(plot_df, x="score", y="value", facet_col="variable")

In [ ]:
import plotly.express as px


result = []
for n_sample, sub_df in df.groupby("N_annotated"):
    result += [{
        "N_annotated": n_sample,
        "Risk type 1": get_risk(sub_df, "error_type_1"),
        "Risk type 1 (support)": get_risk_support(sub_df, "error_type_1"),
        "Risk type 2": get_risk(sub_df, "error_type_2"),
        "Risk type 2 (support)": get_risk_support(sub_df, "error_type_2"),
        "Risk type S": get_risk(sub_df, "error_type_S"),
        "Risk type S (support)": get_risk_support(sub_df, "error_type_S"),
    }]
result = pd.DataFrame(result)
plot_df = result.melt(id_vars=["N_annotated"], value_vars=["Risk type 1", "Risk type 2", "Risk type S"])
px.bar(plot_df, x="N_annotated", y="value", facet_col="variable")

In [ ]:
import plotly.express as px


result = []
df["sampling_method"] = df["sampling_method"].astype(str)
for sampling_method, sub_df in df.groupby("sampling_method"):
    result += [{
        "sampling_method": sampling_method,
        "Risk type 1": get_risk(sub_df, "error_type_1"),
        "Risk type 1 (support)": get_risk_support(sub_df, "error_type_1"),
        "Risk type 2": get_risk(sub_df, "error_type_2"),
        "Risk type 2 (support)": get_risk_support(sub_df, "error_type_2"),
        "Risk type S": get_risk(sub_df, "error_type_S"),
        "Risk type S (support)": get_risk_support(sub_df, "error_type_S"),
    }]
result = pd.DataFrame(result)
plot_df = result.melt(id_vars=["sampling_method"], value_vars=["Risk type 1", "Risk type 2", "Risk type S"])
px.bar(plot_df, x="sampling_method", y="value", facet_col="variable")

In [ ]:
import plotly.express as px


result = []
df["splits_ratio"] = df["splits_ratio"].astype(str)
for splits_ratio, sub_df in df.groupby("splits_ratio"):
    result += [{
        "splits_ratio": splits_ratio,
        "Risk type 1": get_risk(sub_df, "error_type_1"),
        "Risk type 1 (support)": get_risk_support(sub_df, "error_type_1"),
        "Risk type 2": get_risk(sub_df, "error_type_2"),
        "Risk type 2 (support)": get_risk_support(sub_df, "error_type_2"),
        "Risk type S": get_risk(sub_df, "error_type_S"),
        "Risk type S (support)": get_risk_support(sub_df, "error_type_S"),
    }]
result = pd.DataFrame(result)

plot_df = result.melt(id_vars=["splits_ratio"], value_vars=["Risk type 1", "Risk type 2", "Risk type S"])
px.bar(plot_df, x="splits_ratio", y="value", facet_col="variable")

In [ ]:
import plotly.express as px


result = []

for task, sub_df in df.groupby("task"):
    result += [{
        "task": task,
        "Risk type 1": get_risk(sub_df, "error_type_1"),
        "Risk type 1 (support)": get_risk_support(sub_df, "error_type_1"),
        "Risk type 2": get_risk(sub_df, "error_type_2"),
        "Risk type 2 (support)": get_risk_support(sub_df, "error_type_2"),
        "Risk type S": get_risk(sub_df, "error_type_S"),
        "Risk type S (support)": get_risk_support(sub_df, "error_type_S"),
    }]
result = pd.DataFrame(result)

plot_df = result.melt(id_vars=["task"], value_vars=["Risk type 1", "Risk type 2", "Risk type S"])
px.bar(plot_df, x="task", y="value", facet_col="variable")

In [ ]:
import plotly.express as px


result = []

for model_name, sub_df in df.groupby("model_name"):
    result += [{
        "model_name": model_name,
        "Risk type 1": get_risk(sub_df, "error_type_1"),
        "Risk type 1 (support)": get_risk_support(sub_df, "error_type_1"),
        "Risk type 2": get_risk(sub_df, "error_type_2"),
        "Risk type 2 (support)": get_risk_support(sub_df, "error_type_2"),
        "Risk type S": get_risk(sub_df, "error_type_S"),
        "Risk type S (support)": get_risk_support(sub_df, "error_type_S"),
    }]
result = pd.DataFrame(result)

plot_df = result.melt(id_vars=["model_name"], value_vars=["Risk type 1", "Risk type 2", "Risk type S"])
px.bar(plot_df, x="model_name", y="value", facet_col="variable")

In [ ]:
import plotly.express as px


result = []

for hypothesis, sub_df in df.groupby("hypothesis"):
    result += [{
        "hypothesis": hypothesis,
        "Risk type 1": get_risk(sub_df, "error_type_1"),
        "Risk type 1 (support)": get_risk_support(sub_df, "error_type_1"),
        "Risk type 2": get_risk(sub_df, "error_type_2"),
        "Risk type 2 (support)": get_risk_support(sub_df, "error_type_2"),
        "Risk type S": get_risk(sub_df, "error_type_S"),
        "Risk type S (support)": get_risk_support(sub_df, "error_type_S"),
    }]
result = pd.DataFrame(result)

In [ ]:
plot_df = result.melt(id_vars=["hypothesis"], value_vars=["Risk type 1", "Risk type 2", "Risk type S"])

plot_df_R1 = plot_df[plot_df.variable == "Risk type 1"]
px.density_heatmap(plot_df_R1, x="hypothesis", z="value", y="variable", height=500)

In [ ]:
plot_df_R2S = plot_df[plot_df.variable != "Risk type 1"]
px.density_heatmap(plot_df_R2S, x="hypothesis", z="value", y="variable", height=500)